# 07 — Seed-size (θ) and damping (η) scans

Follow-up to notebook 06. Runs from `jobs/open_submit.sh`, all at $\tau = 128$.

**θ scan** (classical Langevin, $\theta$ = 1e-8 … 1e-11): the quantum-bath curve looked like the classical one shifted rigidly by about 10 time units, matching its smaller seed. If some $\theta$ lands on top of `qlangevin`, the bath spectrum doesn't matter, only the seed size does.

**η scan** (quantum bath, Wigner-vacuum start, $\eta$ = 0 … 1; $\eta = 0$ is `twa`): how far back the memory of the initial noise survives as damping is turned on. Expect the overdamped/underdamped crossover near $\eta\,\hat t \sim 1$, i.e. $\eta \sim (2\tau)^{-1/3} \approx 0.16$.

In [ ]:
import sys, os, glob, json
sys.path.insert(0, "..")
import numpy as np
import matplotlib.pyplot as plt

# ---- inputs ----
TAU      = 128
DATA_DIR = "../data/1D_open"
RES_DIR  = "../results/open"
FIGDIR   = "../figures/1d_open"
FONTSIZE = 16

THETA_RUNS = {                        # tag: (label, color)
    "classical":            (r"classical $\theta=10^{-8}$",  "C0"),
    "classical_theta1e-9":  (r"classical $\theta=10^{-9}$",  "C1"),
    "classical_theta1e-10": (r"classical $\theta=10^{-10}$", "C4"),
    "classical_theta1e-11": (r"classical $\theta=10^{-11}$", "C5"),
    "qlangevin":            ("quantum bath (T=0)",           "k"),
}
ETA_RUNS = {                          # tag: eta
    "twa":                0.0,
    "qlv_wigner_eta0.01": 0.01,
    "qlv_wigner_eta0.03": 0.03,
    "qlv_wigner_eta0.1":  0.1,
    "qlv_wigner_eta0.3":  0.3,
    "qlv_wigner_eta1":    1.0,
}

os.makedirs(FIGDIR, exist_ok=True)
res = {}
for tag in list(THETA_RUNS) + list(ETA_RUNS):
    res[tag] = sorted((json.load(open(f)) for f in glob.glob(f"{RES_DIR}/{tag}_tau{TAU}_t*.json")), key=lambda r: r["t"])
    n_chunks = len(glob.glob(f"{DATA_DIR}/{tag}/tau{TAU}/chunk_*.npz"))
    print(f"{tag:>22s}: {n_chunks:2d} chunks, {len(res[tag]):2d} trained times")

## Kink counts

In [ ]:
print(f"{'tag':>22s}  {'mean kinks':>10s}  {'sd':>5s}   N")
for tag in list(THETA_RUNS) + list(ETA_RUNS):
    fs = sorted(glob.glob(f"{DATA_DIR}/{tag}/tau{TAU}/chunk_*.npz"))
    if not fs:
        continue
    n = np.concatenate([np.load(f)["n_defects"] for f in fs])
    print(f"{tag:>22s}  {n.mean():10.2f}  {n.std():5.2f}   {len(n)}")

## θ scan: does a smaller thermal seed reproduce the quantum bath?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(18, 6.5))
for tag, (lab, col) in THETA_RUNS.items():
    if not res[tag]:
        continue
    T = np.array([r["t"] for r in res[tag]])
    ls = "--" if tag == "qlangevin" else "-"
    ax[0].plot(T, [r["sign_err"]  for r in res[tag]], "o" + ls, color=col, ms=8, lw=2, label=lab)
    ax[1].plot(T, [r["count_acc"] for r in res[tag]], "o" + ls, color=col, ms=8, lw=2, label=lab)

# ---- reference lines (edit freely) ----
for a in ax:
    a.axvline(0.0,  color="grey", ls=":", lw=1.5)
    a.axvline(16.0, color="grey", ls=":", lw=1.5)
ax[0].axhline(0.5, color="grey", ls="-", lw=1.0)

ax[0].set_ylabel("per-site sign error (RNN)", fontsize=FONTSIZE)
ax[1].set_ylabel("defect-count accuracy", fontsize=FONTSIZE)
ax[0].set_ylim(0, 0.55); ax[1].set_ylim(0, 1.05)
for a in ax:
    a.set_xlim(-40, 70)
    a.set_xlabel(r"input time $t$", fontsize=FONTSIZE)
    a.tick_params(labelsize=FONTSIZE - 2)
    a.legend(fontsize=FONTSIZE - 3)
fig.suptitle(rf"$\theta$ scan vs quantum bath, $\tau_Q = {TAU}$", fontsize=FONTSIZE + 2)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/theta_scan_tau{TAU}.png", dpi=300)
plt.show()

In [ ]:
# onset time vs log10(theta): if lock-in is set by seed size, this is a straight line
# ---- inputs ----
THRESH = [0.4, 0.2, 0.1]
THETAS = {"classical": 1e-8, "classical_theta1e-9": 1e-9, "classical_theta1e-10": 1e-10, "classical_theta1e-11": 1e-11}

plt.figure(figsize=(9, 6))
for th, m in zip(THRESH, ["o", "s", "^"]):
    x, y = [], []
    for tag, theta in THETAS.items():
        T = np.array([r["t"] for r in res[tag]]); E = np.array([r["sign_err"] for r in res[tag]])
        if (E < th).any():
            x.append(theta); y.append(T[E < th].min())
    plt.semilogx(x, y, m + "-", ms=10, lw=2, label=rf"err $< {th}$")
    Tq = np.array([r["t"] for r in res["qlangevin"]]); Eq = np.array([r["sign_err"] for r in res["qlangevin"]])
    if (Eq < th).any():
        plt.axhline(Tq[Eq < th].min(), color=plt.gca().lines[-1].get_color(), ls="--", lw=1.2)
plt.xlim(1e-11, 1e-8)
plt.xlabel(r"$\theta$", fontsize=FONTSIZE)
plt.ylabel("first input time below threshold", fontsize=FONTSIZE)
plt.title("dashed: quantum bath", fontsize=FONTSIZE)
plt.tick_params(labelsize=FONTSIZE - 2)
plt.legend(fontsize=FONTSIZE - 2)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/theta_onset_tau{TAU}.png", dpi=300)
plt.show()

## η scan: how much damping erases the memory of the initial noise?

In [ ]:
colors = plt.cm.viridis(np.linspace(0, 0.9, len(ETA_RUNS)))
plt.figure(figsize=(12, 7))
for (tag, eta), col in zip(ETA_RUNS.items(), colors):
    if not res[tag]:
        continue
    T = np.array([r["t"] for r in res[tag]])
    plt.plot(T, [r["sign_err"] for r in res[tag]], "o-", color=col, ms=8, lw=2, label=rf"$\eta = {eta:g}$")

# ---- reference lines (edit freely) ----
plt.axhline(0.5, color="grey", ls="-", lw=1.0)
plt.axvline(0.0, color="grey", ls=":", lw=1.5)

plt.xlim(-185, 70)
plt.ylim(0, 0.55)
plt.xlabel(r"input time $t$", fontsize=FONTSIZE)
plt.ylabel("per-site sign error (RNN)", fontsize=FONTSIZE)
plt.title(rf"quantum bath + Wigner start, $\tau_Q = {TAU}$", fontsize=FONTSIZE)
plt.tick_params(labelsize=FONTSIZE - 2)
plt.legend(fontsize=FONTSIZE - 2)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/eta_scan_tau{TAU}.png", dpi=300)
plt.show()

In [ ]:
# memory vs damping: sign error at fixed early input times, as a function of eta
# ---- inputs ----
T_FIXED = [-150, -60, -20]

etas = np.array([e for e in ETA_RUNS.values() if e > 0])
plt.figure(figsize=(9, 6))
for t, m in zip(T_FIXED, ["o", "s", "^"]):
    y = [next((r["sign_err"] for r in res[tag] if r["t"] == t), np.nan) for tag, e in ETA_RUNS.items() if e > 0]
    plt.semilogx(etas, y, m + "-", ms=10, lw=2, label=rf"$t = {t}$")
    y0 = next((r["sign_err"] for r in res["twa"] if r["t"] == t), np.nan)
    plt.axhline(y0, color=plt.gca().lines[-1].get_color(), ls="--", lw=1.2)

# ---- reference lines (edit freely) ----
plt.axhline(0.5, color="grey", ls="-", lw=1.0)
plt.axvline(0.16, color="grey", ls=":", lw=1.5)
plt.text(0.17, 0.05, r"$\eta\,\hat t_{\rm TWA} = 1$", fontsize=FONTSIZE - 2, color="grey")

plt.xlim(etas.min(), etas.max())
plt.ylim(0, 0.55)
plt.xlabel(r"damping $\eta$", fontsize=FONTSIZE)
plt.ylabel("per-site sign error (RNN)", fontsize=FONTSIZE)
plt.title(r"dashed: $\eta = 0$ (closed TWA)", fontsize=FONTSIZE)
plt.tick_params(labelsize=FONTSIZE - 2)
plt.legend(fontsize=FONTSIZE - 2)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/memory_vs_eta_tau{TAU}.png", dpi=300)
plt.show()

In [ ]:
# initial-state check at eta = 1: zero start (qlangevin) vs Wigner start (qlv_wigner_eta1)
plt.figure(figsize=(10, 6))
for tag, col, lab in [("qlangevin", "k", r"$\eta=1$, zero start"), ("qlv_wigner_eta1", "C3", r"$\eta=1$, Wigner start")]:
    if not res[tag]:
        continue
    T = np.array([r["t"] for r in res[tag]])
    plt.plot(T, [r["sign_err"] for r in res[tag]], "o-", color=col, ms=8, lw=2, label=lab)
plt.axhline(0.5, color="grey", ls="-", lw=1.0)
plt.xlim(-185, 70)
plt.ylim(0, 0.55)
plt.xlabel(r"input time $t$", fontsize=FONTSIZE)
plt.ylabel("per-site sign error (RNN)", fontsize=FONTSIZE)
plt.tick_params(labelsize=FONTSIZE - 2)
plt.legend(fontsize=FONTSIZE - 2)
plt.tight_layout()
plt.show()